<a href="https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
%pip -q install duckdb

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Connect to the warehouse (token read from Colab's safe box)
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Same frame as ML-04 to ML-09: one row per page. Clues 1-15 March, outcome 16-31 March.
pages = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS days_visible_h1,
        SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15') AS impressions_h2,
        SUM(gsc_clicks)      FILTER (WHERE report_date > DATE '2026-03-15') AS clicks_h2,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h2
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING impressions_h1 >= 100 AND impressions_h2 >= 50
""").df()
pages = pages.dropna(subset=["position_h1", "position_h2"]).copy()
pages = pages.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)  # fixed order -> reproducible

bands = [0, 3, 5, 7, 10, 15, 20, 30, 50, 1000]
pages["ctr_h1"] = pages["clicks_h1"] / pages["impressions_h1"] * 100
pages["band_h1"] = pd.cut(pages["position_h1"], bands)
pages["expected_ctr_h1"] = pages.groupby("band_h1", observed=True)["ctr_h1"].transform("median")
pages["missed_clicks_h1"] = pages["impressions_h1"] * (pages["expected_ctr_h1"] - pages["ctr_h1"]) / 100

# Outcome (16-31 March) - used ONLY to check the playbook, never to rank
pages["ctr_h2"] = pages["clicks_h2"] / pages["impressions_h2"] * 100
pages["band_h2"] = pd.cut(pages["position_h2"], bands)
pages["expected_ctr_h2"] = pages.groupby("band_h2", observed=True)["ctr_h2"].transform("median")
pages["still_under"] = (pages["ctr_h2"] < pages["expected_ctr_h2"]).astype(int)

# The frozen Week-4 rule decides WHO is in the queue and IN WHAT ORDER
MIN_IMPRESSIONS, MAX_POSITION = 1000, 20
flagged = (pages["impressions_h1"] >= MIN_IMPRESSIONS) & (pages["position_h1"] <= MAX_POSITION) & (pages["ctr_h1"] < pages["expected_ctr_h1"])
pages["flagged"] = flagged
pages["score"] = np.where(flagged, pages["missed_clicks_h1"], 0.0)
FEATURES = ["impressions_h1", "ctr_h1", "position_h1", "days_visible_h1", "missed_clicks_h1"]

# Second opinion: random-forest probability that the problem lasts, scored by a model that never saw the page's client
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
folds = list(GroupKFold(n_splits=5).split(pages, groups=pages["client_hash_id"]))
pages["persistence_prob"] = np.nan
for train_idx, test_idx in folds:
    train, test = pages.iloc[train_idx], pages.iloc[test_idx]
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1)
    rf.fit(train[FEATURES], train["still_under"])
    pages.loc[test.index, "persistence_prob"] = rf.predict_proba(test[FEATURES])[:, 1]
pages["confidence"] = pd.cut(pages["persistence_prob"], [-0.01, 0.6, 0.8, 1.0], labels=["low", "medium", "high"]).astype(str)

# The queue: flagged pages, biggest estimated click gap first
queue = pages[pages["flagged"]].sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
top100_share = queue.head(100)["client_hash_id"].value_counts() / 100
dominant_clients = set(top100_share[top100_share >= 0.20].index)

# Action: data problems first, then tracking, otherwise a title/meta review
queue["action"] = np.select(
    [queue["position_h1"] < 1, queue["clicks_h1"] == 0],
    ["check_data_quality", "check_tracking_first"],
    default="review_title_and_meta")

def reason_codes(row):
    codes = ["low_ctr_for_position"]
    if row["clicks_h1"] == 0: codes.append("zero_clicks")
    if row["position_h1"] < 1: codes.append("impossible_position")
    if row["client_hash_id"] in dominant_clients: codes.append("dominant_client")
    return "; ".join(codes)
queue["reason_codes"] = queue.apply(reason_codes, axis=1)
queue["client"] = queue["client_hash_id"].str[-6:]

print(f"Pages scored: {len(pages):,} | flagged into the queue: {len(queue):,} | clients: {pages['client_hash_id'].nunique()}")
print("Clients holding at least 20% of the top 100:", len(dominant_clients))
print("\nQueue by action (still_under_rate = share still below typical CTR in 16-31 March, observed afterwards):")
display(queue.groupby("action").agg(pages=("rank", "size"), missed_clicks=("score", "sum"),
                                    still_under_rate=("still_under", "mean")).round(2))
show = ["rank", "client", "impressions_h1", "clicks_h1", "ctr_h1", "expected_ctr_h1", "position_h1",
        "score", "action", "reason_codes", "confidence"]
print("\nTop 15 of the queue:")
display(queue[show].head(15).round(2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages scored: 75,817 | flagged into the queue: 8,710 | clients: 37
Clients holding at least 20% of the top 100: 2

Queue by action (still_under_rate = share still below typical CTR in 16-31 March, observed afterwards):


,pages,missed_clicks,still_under_rate
action,,,
check_data_quality,194,1154.49,0.78
check_tracking_first,1625,5081.30,0.67
review_title_and_meta,6891,25938.07,0.67



Top 15 of the queue:


,rank,client,impressions_h1,clicks_h1,ctr_h1,expected_ctr_h1,position_h1,score,action,reason_codes,confidence
0,1,f265ea,83772.0,0.0,0.00,0.25,0.11,205.68,check_data_quality,low_ctr_for_position; zero_clicks; impossible_...,high
1,2,5e0096,73639.0,18.0,0.02,0.25,2.95,162.80,review_title_and_meta,low_ctr_for_position; dominant_client,high
2,3,f265ea,58553.0,1.0,0.00,0.24,4.75,142.19,review_title_and_meta,low_ctr_for_position; dominant_client,high
3,4,5e0096,52378.0,18.0,0.03,0.24,4.65,110.09,review_title_and_meta,low_ctr_for_position; dominant_client,high
4,5,5e0096,86860.0,51.0,0.06,0.17,5.95,93.17,review_title_and_meta,low_ctr_for_position; dominant_client,high
5,6,9f63c4,55680.0,15.0,0.03,0.17,6.68,77.41,review_title_and_meta,low_ctr_for_position,high
6,7,5e0096,83715.0,133.0,0.16,0.24,3.52,71.72,review_title_and_meta,low_ctr_for_position; dominant_client,high
7,8,5e0096,49314.0,2.0,0.00,0.14,8.42,67.95,review_title_and_meta,low_ctr_for_position; dominant_client,high
8,9,5e0096,27715.0,0.0,0.00,0.24,4.45,67.77,check_tracking_first,low_ctr_for_position; zero_clicks; dominant_cl...,high
9,10,043229,33020.0,14.0,0.04,0.25,1.60,67.07,review_title_and_meta,low_ctr_for_position,high


In [2]:
# ARCHETYPE -> ACTION: every page sorted into one simple type (rule-based types, not clusters)
below = pages["ctr_h1"] < pages["expected_ctr_h1"]
pages["archetype"] = np.select(
    [pages["flagged"] & (pages["position_h1"] < 1),
     pages["flagged"] & (pages["clicks_h1"] == 0),
     pages["flagged"],
     below & (pages["position_h1"] > MAX_POSITION),
     below],
    ["C. Broken position data", "B. Silent page (0 clicks)", "A. Visible under-performer",
     "E. Low CTR, deep position (>20)", "D. Low CTR, low volume (<1,000 impr.)"],
    default="F. At or above typical CTR")

action_for = {
    "A. Visible under-performer": "review_title_and_meta",
    "B. Silent page (0 clicks)": "check_tracking_first",
    "C. Broken position data": "check_data_quality",
    "D. Low CTR, low volume (<1,000 impr.)": "monitor",
    "E. Low CTR, deep position (>20)": "out_of_scope_for_ctr_fix",
    "F. At or above typical CTR": "no_action",
}
arche = pages.groupby("archetype").agg(
    pages=("still_under", "size"),
    missed_clicks=("missed_clicks_h1", lambda s: s.clip(lower=0).sum()),
    still_under_rate=("still_under", "mean")).round(2)
arche.insert(0, "action", [action_for[a] for a in arche.index])
display(arche)

,action,pages,missed_clicks,still_under_rate
archetype,,,,
A. Visible under-performer,review_title_and_meta,6891,25938.07,0.67
B. Silent page (0 clicks),check_tracking_first,1625,5081.30,0.67
C. Broken position data,check_data_quality,194,1154.49,0.78
"D. Low CTR, low volume (<1,000 impr.)",monitor,21680,10630.68,0.57
"E. Low CTR, deep position (>20)",out_of_scope_for_ctr_fix,3598,1004.24,0.12
F. At or above typical CTR,no_action,41829,0.00,0.20


In [3]:
# DECAY INSIGHT: content age at the decision moment (16 March 2026) vs CTR problems
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
created = con.sql(f"SELECT client_hash_id, content_hash_id, content_created_date FROM {CONTENT}").df()
aged = pages.merge(created, on=["client_hash_id", "content_hash_id"], how="left")
aged["age_days"] = (pd.Timestamp("2026-03-16") - pd.to_datetime(aged["content_created_date"])).dt.days
aged = aged[aged["age_days"] >= 0]
aged["age_bucket"] = pd.cut(aged["age_days"], [-1, 90, 180, 270, 365, 100000],
                            labels=["0-90 d", "91-180 d", "181-270 d", "271-365 d", "365+ d"])
decay = aged.groupby("age_bucket", observed=True).agg(
    pages=("still_under", "size"),
    share_below_typical_ctr=("ctr_h1", lambda s: (s < aged.loc[s.index, "expected_ctr_h1"]).mean()),
    share_flagged=("flagged", "mean"),
    still_under_rate=("still_under", "mean")).round(3)
print("Pages with a known creation date:", len(aged), "of", len(pages))
display(decay)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages with a known creation date: 75817 of 75817


,pages,share_below_typical_ctr,share_flagged,still_under_rate
age_bucket,,,,
0-90 d,24377,0.442,0.110,0.382
91-180 d,14333,0.464,0.123,0.367
181-270 d,19831,0.430,0.119,0.325
271-365 d,9575,0.444,0.102,0.321
365+ d,7701,0.493,0.120,0.361


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.